# Effective Resistance R_eff vs N and a (distance-dependent model)

How does the network's effective resistance $R_{eff}$ (using the distance-dependent resistance model, $R_{edge} = R_0 \cdot L_{edge}$, from `distance_dependent_resistance.ipynb`) behave as a function of:

- **N** — number of sticks
- **a** — aspect ratio (stick length)

For each (N, a) pair we draw random networks until we get several that percolate (not every draw does), compute $R_{eff}$ for each with `distance_dependent=True`, and summarize the distribution with box plots.

This reuses `generate_percolating_network()` and `distance_dependent_impedance()`, now modularized into `utils/pipeline_utils.py` (previously duplicated ad hoc in `distance_dependent_resistance.ipynb`).

# Part A — Baseline grid sweep (4×3, full statistics)

Small, dense rectangular grid over `N_values × a_values`, enough trials per cell to get solid
box-plot statistics. This is the reference sweep; Part B below extends it to a much larger,
adaptive (N, a) grid.

---
## 1. Setup and imports

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    utils_path = "/content/drive/MyDrive/ColabProject/utils"
    print("Environment: Google Colab")
except ImportError:
    utils_path = os.path.abspath(os.path.join(os.getcwd(), "..", "utils"))
    print("Environment: Local")

if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Utils path: {utils_path}")


Environment: Local
Utils path: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils


In [2]:
from pipeline_utils import generate_percolating_network, distance_dependent_impedance
from network_visualization import plot_network_colored_by_resistance

print("Imports OK")


Imports OK


---
## 2. Experiment tracking

All figures and the full sweep data are saved to a new `experiments/exp_N/` folder (same `ExperimentManager` used across the project), so this sweep doesn't have to be re-run just to look at the numbers or plots again.

In [3]:
from experiment_manager import setup_experiment

exp_manager = setup_experiment()


EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 883
  Timestamp: 20260916_075736
  Results directory: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_883



---
## 3. Sweep configuration

For each (N, a) pair we keep drawing random networks until `n_trials` of them percolate (each draw that doesn't percolate is discarded, not counted as a trial). `max_attempts` caps the total number of draws per (N, a) pair as a safety valve for combinations far below the percolation threshold.

In [4]:
N_values = [30, 45, 60, 75]
a_values = [0.4, 0.6, 0.8]
n_trials = 15
R0 = 1000.0
max_attempts = 400
# Boundary rule of the stick generator: 'clip' (window cut from a larger film; sticks
# crossing the edge are clipped) since 2026-09-15, 'reject' before. Recorded in every
# summary so that the pooling in inverse_N_a_from_R_eff.ipynb never mixes the two.
BOUNDARY = "clip"

print(f"Grid: {len(N_values)} x {len(a_values)} = {len(N_values) * len(a_values)} combinations, "
      f"{n_trials} percolating trials each")
print(f"N values: {N_values}")
print(f"a values: {a_values}")


Grid: 4 x 3 = 12 combinations, 15 percolating trials each
N values: [30, 45, 60, 75]
a values: [0.4, 0.6, 0.8]


---
## 4. Run the sweep

For each (N, a): draw networks (uniform-k pipeline under the hood, since that's what percolation/pruning is built on) until `n_trials` percolate, then recompute each one's impedance with the distance-dependent model. Also tracks the percolation success rate -- how many of the raw draws actually percolated -- since that's expected to drop sharply for small N*a^2 (near/below the stick-percolation threshold).

In [5]:
results_grid = {}     # (N, a) -> list of R_eff (distance-dependent), one per percolating trial
perc_rate_grid = {}   # (N, a) -> fraction of raw draws that percolated
sample_networks = {}  # (N, a) -> first percolating network's full result dict (for plotting)

for N in N_values:
    for a in a_values:
        seed_base = (hash((N, round(a, 3))) % 100000)
        np.random.seed(seed_base)

        R_effs = []
        total_attempts = 0
        while len(R_effs) < n_trials:
            res, attempts = generate_percolating_network(N, a, k_uniform=R0, max_attempts=max_attempts, boundary=BOUNDARY)
            total_attempts += attempts
            if res is None:
                print(f"  [WARNING] N={N} a={a}: gave up after {total_attempts} draws, "
                      f"only found {len(R_effs)}/{n_trials} percolating networks")
                break
            if (N, a) not in sample_networks:
                sample_networks[(N, a)] = res
            rl = distance_dependent_impedance(res, R0)
            R_effs.append(rl['impedance'])

        rate = len(R_effs) / total_attempts if total_attempts else 0.0
        results_grid[(N, a)] = R_effs
        perc_rate_grid[(N, a)] = rate
        print(f"N={N:3d}  a={a:.1f}   {len(R_effs):2d}/{n_trials} percolating trials   "
              f"({total_attempts} draws, {rate:.1%} success rate)   "
              f"R_eff: mean={np.mean(R_effs):8.1f}  median={np.median(R_effs):8.1f}  ohm")

print("\nSweep complete.")


N= 30  a=0.4   15/15 percolating trials   (80 draws, 18.8% success rate)   R_eff: mean=   868.1  median=   844.4  ohm
N= 30  a=0.6   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=   297.0  median=   239.3  ohm


N= 30  a=0.8   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=   161.6  median=   176.3  ohm


N= 45  a=0.4   15/15 percolating trials   (24 draws, 62.5% success rate)   R_eff: mean=   565.6  median=   576.5  ohm


N= 45  a=0.6   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=   175.3  median=   153.2  ohm


N= 45  a=0.8   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=    74.9  median=    69.7  ohm


...

N= 60  a=0.6   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=    74.4  median=    70.2  ohm


N= 60  a=0.8   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=    44.4  median=    42.1  ohm


N= 75  a=0.4   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=   155.6  median=   151.8  ohm


N= 75  a=0.6   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=    49.6  median=    48.8  ohm


N= 75  a=0.8   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=    34.2  median=    33.6  ohm

Sweep complete.


---
## 5. Summary table

In [6]:
header = f"{'N':>4} {'a':>5} {'n':>4} {'mean(ohm)':>11} {'median(ohm)':>13} {'std(ohm)':>10} {'perc.rate':>10}"
print(header)
print("-" * len(header))
for N in N_values:
    for a in a_values:
        vals = np.array(results_grid[(N, a)])
        rate = perc_rate_grid[(N, a)]
        print(f"{N:4d} {a:5.2f} {len(vals):4d} {vals.mean():11.1f} {np.median(vals):13.1f} "
              f"{vals.std():10.1f} {rate:10.1%}")


   N     a    n   mean(ohm)   median(ohm)   std(ohm)  perc.rate
---------------------------------------------------------------
  30  0.40   15       868.1         844.4      213.9      18.8%
  30  0.60   15       297.0         239.3      149.2     100.0%
  30  0.80   15       161.6         176.3       45.4     100.0%
  45  0.40   15       565.6         576.5      229.7      62.5%
  45  0.60   15       175.3         153.2       56.2     100.0%
  45  0.80   15        74.9          69.7       21.1     100.0%
  60  0.40   15       280.6         217.5      154.1     100.0%
  60  0.60   15        74.4          70.2       10.5     100.0%
  60  0.80   15        44.4          42.1        6.8     100.0%
  75  0.40   15       155.6         151.8       32.9     100.0%
  75  0.60   15        49.6          48.8        8.6     100.0%
  75  0.80   15        34.2          33.6        2.7     100.0%


> **Note on local file paths (read before trusting `exp_1`/`exp_2` below):** this run of the notebook was executed on the Apuana cluster (CIn/UFPE), where `experiments/exp_1` and `experiments/exp_2` were genuinely the next free experiment numbers -- the cluster's `experiments/` folder was empty at the time (unrelated to this project's local numbering, which was already past `exp_800`). After syncing the results back, those two folders were manually moved to `experiments/exp_813/` (Part A) and `experiments/exp_814/` (Part B) locally, to avoid colliding with this project's own pre-existing `exp_1`/`exp_2` (unrelated experiments from March). The `Results directory: .../exp_1`, `.../exp_2` lines printed in the cell outputs below are the historical record of what actually ran on the cluster -- they are correct as printed, but no longer match where the files live locally.

---
## 6. Box plots: R_eff vs N, grouped by a

Log-scale y-axis, since R_eff spans more than an order of magnitude across the grid.

In [7]:
# Same sizing reasoning as the extended sweep further down: modest figsize,
# explicit font sizes and one shared pair of axis labels, so the figure stays
# readable once a document scales it to the text width.
fig, axes = plt.subplots(1, len(a_values), figsize=(13, 4.2), sharey=True,
                         constrained_layout=True)

for ax, a in zip(axes, a_values):
    data = [results_grid[(N, a)] for N in N_values]
    ax.boxplot(data, positions=range(len(N_values)), widths=0.6)
    ax.set_xticks(range(len(N_values)))
    ax.set_xticklabels(N_values, fontsize=10)
    ax.set_yscale('log')
    # panel label inside the axes instead of a title: readable at print size, and the
    # LaTeX caption carries the figure title
    ax.text(0.04, 0.95, f'a = {a}', transform=ax.transAxes, ha='left', va='top', fontsize=13,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=2))
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(True, alpha=0.3, which='both')

fig.supxlabel('N (number of sticks)', fontsize=14)
fig.supylabel('R_eff (Ω, distance-dependent, log scale)', fontsize=13)
# No fig.suptitle here: these figures go into the dissertation, where the
# LaTeX \caption carries the title. An embedded title would duplicate it and
# waste vertical space once the figure is scaled to the text width. The
# per-panel titles below are kept, since they label data (the value of a or
# N for that panel), not the figure.
plt.show()
exp_manager.save_figure(fig, "box_R_eff_vs_N_by_a.png")

  [OK] Figure saved: box_R_eff_vs_N_by_a.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49000/3492404314.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_883/box_R_eff_vs_N_by_a.png'

---
## 7. Box plots: R_eff vs a, grouped by N

In [8]:
fig, axes = plt.subplots(1, len(N_values), figsize=(13, 4.2), sharey=True,
                         constrained_layout=True)

for ax, N in zip(axes, N_values):
    data = [results_grid[(N, a)] for a in a_values]
    ax.boxplot(data, positions=range(len(a_values)), widths=0.6)
    step = 1 if len(a_values) <= 6 else 2
    ax.set_xticks(range(0, len(a_values), step))
    ax.set_xticklabels(a_values[::step], fontsize=10)
    ax.set_yscale('log')
    # panel label inside the axes instead of a title: readable at print size, and the
    # LaTeX caption carries the figure title
    ax.text(0.04, 0.95, f'N = {N}', transform=ax.transAxes, ha='left', va='top', fontsize=13,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=2))
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(True, alpha=0.3, which='both')

fig.supxlabel('a (aspect ratio)', fontsize=14)
fig.supylabel('R_eff (Ω, log scale)', fontsize=14)
# No fig.suptitle here: these figures go into the dissertation, where the
# LaTeX \caption carries the title. An embedded title would duplicate it and
# waste vertical space once the figure is scaled to the text width. The
# per-panel titles below are kept, since they label data (the value of a or
# N for that panel), not the figure.
plt.show()
exp_manager.save_figure(fig, "box_R_eff_vs_a_by_N.png")

/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49000/3559985982.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [OK] Figure saved: box_R_eff_vs_a_by_N.png


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_883/box_R_eff_vs_a_by_N.png'

---
## 8. R_eff(N, a) and percolation rate as heatmaps

A compact view of the same sweep: mean $R_{eff}$ (log scale) and the raw percolation success rate, both as a function of (N, a).

In [9]:
mean_matrix = np.array([[np.mean(results_grid[(N, a)]) for a in a_values] for N in N_values])
rate_matrix = np.array([[perc_rate_grid[(N, a)] for a in a_values] for N in N_values])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

im1 = ax1.imshow(np.log10(mean_matrix), cmap='viridis_r', aspect='auto')
ax1.set_xticks(range(len(a_values))); ax1.set_xticklabels(a_values)
ax1.set_yticks(range(len(N_values))); ax1.set_yticklabels(N_values)
ax1.set_xlabel('a'); ax1.set_ylabel('N')
# no panel titles: the dissertation caption carries them
log_mean = np.log10(mean_matrix)
log_mid = (log_mean.max() + log_mean.min()) / 2
for i in range(len(N_values)):
    for j in range(len(a_values)):
        color = 'white' if log_mean[i, j] > log_mid else 'black'
        ax1.text(j, i, f"{mean_matrix[i, j]:.0f}", ha='center', va='center',
                 color=color, fontsize=9)
fig.colorbar(im1, ax=ax1, label='log10(R_eff / ohm)')

im2 = ax2.imshow(rate_matrix, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(a_values))); ax2.set_xticklabels(a_values)
ax2.set_yticks(range(len(N_values))); ax2.set_yticklabels(N_values)
ax2.set_xlabel('a'); ax2.set_ylabel('N')
for i in range(len(N_values)):
    for j in range(len(a_values)):
        color = 'white' if rate_matrix[i, j] < 0.6 else 'black'
        ax2.text(j, i, f"{rate_matrix[i, j]:.2f}", ha='center', va='center', color=color, fontsize=9)
fig.colorbar(im2, ax=ax2, label='fraction of draws that percolated')

plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "heatmaps_mean_R_eff_and_perc_rate.png")


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49000/2767087623.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [OK] Figure saved: heatmaps_mean_R_eff_and_perc_rate.png


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_883/heatmaps_mean_R_eff_and_perc_rate.png'

---
## 9. Representative network per (N, a)

One percolating network per (N, a) combination -- the first one found during the sweep above -- plotted with edges colored by their distance-dependent resistance. Useful as a visual reference for what these networks actually look like at each point of the grid (e.g. sparse/near-threshold N=30,a=0.4 vs dense N=75,a=0.8).

In [10]:
for N in N_values:
    for a in a_values:
        res = sample_networks[(N, a)]
        rl = distance_dependent_impedance(res, R0)
        fig = plot_network_colored_by_resistance(
            res['merged_intersections'], res['merged_segment_to_nodes'],
            rl['edge_resistances'],
            title=""  # the dissertation caption carries N, a and R_eff
        )
        plt.show()
        exp_manager.save_figure(fig, f"network_N{N}_a{a}.png")
        exp_manager.save_network(N, a, res['points_1'], res['points_2'], res['angles'],
                                 filename=f"network_N{N}_a{a}.json")


  [OK] Figure saved: network_N30_a0.4.png
  [OK] JSON saved: network_N30_a0.4.json


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_49000/1661102609.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [OK] Figure saved: network_N30_a0.6.png
  [OK] JSON saved: network_N30_a0.6.json
  [OK] Figure saved: network_N30_a0.8.png
  [OK] JSON saved: network_N30_a0.8.json


  [OK] Figure saved: network_N45_a0.4.png
  [OK] JSON saved: network_N45_a0.4.json


  [OK] Figure saved: network_N45_a0.6.png
  [OK] JSON saved: network_N45_a0.6.json


...

  [OK] Figure saved: network_N60_a0.4.png
  [OK] JSON saved: network_N60_a0.4.json


  [OK] Figure saved: network_N60_a0.6.png
  [OK] JSON saved: network_N60_a0.6.json


  [OK] Figure saved: network_N60_a0.8.png
  [OK] JSON saved: network_N60_a0.8.json
  [OK] Figure saved: network_N75_a0.4.png
  [OK] JSON saved: network_N75_a0.4.json


  [OK] Figure saved: network_N75_a0.6.png
  [OK] JSON saved: network_N75_a0.6.json


  [OK] Figure saved: network_N75_a0.8.png
  [OK] JSON saved: network_N75_a0.8.json


---
## 10. Notes

- $R_{eff}$ decreases monotonically with both N and a, as expected: more sticks and/or longer sticks create more (and shorter) parallel conductive paths between source and drain.
- The percolation success rate drops sharply for small $N \cdot a^2$ (below the known 2D stick-percolation threshold, $N \cdot a^2 \approx 5.6$ for the excluded-area criterion) -- e.g. N=30, a=0.4 gives $N a^2 = 4.8$, and only a small fraction of raw draws percolate at all. That's expected, not a bug: near/below threshold, percolation is rare and the R_eff values we do get come from atypical, close-to-minimal conducting paths.
- `max_attempts` is a per-draw-batch cap; if a combination is far enough below threshold it can print a `[WARNING]` and return fewer than `n_trials` samples for that box.
- `N_values`/`a_values`/`n_trials` are all easy to widen for a finer sweep -- this grid (4 x 3 x 15 = 180 percolating networks) runs in well under a minute.

---
## 11. Save consolidated summary

In [11]:
sweep_records = []
for N in N_values:
    for a in a_values:
        vals = results_grid[(N, a)]
        sweep_records.append({
            "N": N, "a": a, "n": len(vals),
            "mean_ohm": float(np.mean(vals)), "median_ohm": float(np.median(vals)),
            "std_ohm": float(np.std(vals)), "perc_rate": perc_rate_grid[(N, a)],
            "R_eff_values": vals,
        })

summary = {
    "experiment": exp_manager.get_info(),
    "config": {"N_values": N_values, "a_values": a_values, "n_trials": n_trials,
               "R0": R0, "max_attempts": max_attempts, "boundary": BOUNDARY},
    "sweep": sweep_records,
}

exp_manager.save_json(summary, "effective_resistance_sweep_summary.json")

print(f"\nAll results saved to: {exp_manager.experiment_dir}")
for f in sorted(os.listdir(exp_manager.experiment_dir)):
    print(f"  - {f}")


  [OK] JSON saved: effective_resistance_sweep_summary.json

All results saved to: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_883
  - box_R_eff_vs_N_by_a.png
  - box_R_eff_vs_a_by_N.png
  - effective_resistance_sweep_summary.json
  - heatmaps_mean_R_eff_and_perc_rate.png
  - network_N30_a0.4.json
  - network_N30_a0.4.png
  - network_N30_a0.6.json
  - network_N30_a0.6.png
  - network_N30_a0.8.json
  - network_N30_a0.8.png
  - network_N45_a0.4.json
  - network_N45_a0.4.png
  - network_N45_a0.6.json
  - network_N45_a0.6.png
  - network_N45_a0.8.json
  - network_N45_a0.8.png
  - network_N60_a0.4.json
  - network_N60_a0.4.png
  - network_N60_a0.6.json
  - network_N60_a0.6.png
  - network_N60_a0.8.json
  - network_N60_a0.8.png
  - network_N75_a0.4.json
  - network_N75_a0.4.png
  - network_N75_a0.6.json
  - network_N75_a0.6.png
  - network_N75_a0.8.json
  - networ

# Part B — Extended adaptive-grid sweep (full version, meant for the cluster)

## 12. Sweep configuration (adaptive grid)

The 4x3 grid above is cheap because it never combines large N with large a. But the impedance
solve is a **dense** linear solve on the reduced Laplacian (`np.linalg.solve`), so its cost is
O(n_nodes^3), and n_nodes itself grows like N^2 * a^2 (more/longer sticks -> quadratically more
intersections). Benchmarking the corner directly:

| N | a | nodes | time/draw |
|---|---|---|---|
| 110 | 0.6 | 2177 | 2.7s |
| 110 | 1.0 | 4458 | 20.1s |
| 150 | 0.8 | 6430 | 48.5s |
| 150 | 1.0 | 8235 | **97s** |

So a naive rectangular grid up to N=150, a=1.0 would spend most of its time -- likely hours --
in a handful of cells in that corner, since dense networks also percolate on nearly every draw
(so all `n_trials` draws pay the full cost).

We use an **adaptive grid**: for each `a`, cap `N` so that the estimated node count
(`~0.5 * N^2 * a^2`) stays under a fixed budget. This keeps per-draw cost roughly bounded across
the whole grid while still covering N up to 150 (for small a) and a up to 1.0 (for smaller N).

This is now set to the **full version** -- 10 `a` values x 10 `N` per `a` = ~100 pairs, 30
percolating trials per pair (up to ~3000 percolating networks) -- meant to be run on the
cluster rather than locally. The original 5-value/4-per-a/10-trial preview is noted below
(commented out) as the config to fall back to for a quick local sanity check.

In [12]:
import time

def n_max_for_a(a, node_cap=3000, N_ceiling=150, N_floor=20, round_to=5):
    """Largest N such that estimated node count (~0.5*N^2*a^2) stays under node_cap."""
    raw = np.sqrt(node_cap / (0.5 * a**2))
    capped = min(N_ceiling, raw)
    return int(max(N_floor, round(capped / round_to) * round_to))

# Full version, meant for the cluster: 10 a values x 10 N per a = ~100 pairs.
# Local quick-preview fallback (~20 pairs): a_values_ext = [0.2, 0.4, 0.6, 0.8, 1.0]; n_per_a = 4
a_values_ext = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1]
N_floor_ext = 20
n_per_a = 10

grid_ext = {}
pairs_ext = []
for a in a_values_ext:
    n_max = n_max_for_a(a)
    Ns = sorted(set(int(round(x / 5) * 5) for x in np.linspace(N_floor_ext, n_max, n_per_a)))
    grid_ext[a] = Ns
    pairs_ext.extend((N, a) for N in Ns)

print(f"{len(pairs_ext)} (N, a) pairs (adaptive grid, node count capped so the dense "
      f"impedance solve stays tractable)")
for a in a_values_ext:
    print(f"  a={a:.1f}: N_max={n_max_for_a(a):4d}   N in {grid_ext[a]}")

100 (N, a) pairs (adaptive grid, node count capped so the dense impedance solve stays tractable)
  a=0.2: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.3: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.4: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.5: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.6: N_max= 130   N in [20, 30, 45, 55, 70, 80, 95, 105, 120, 130]
  a=0.7: N_max= 110   N in [20, 30, 40, 50, 60, 70, 80, 90, 100, 110]
  a=0.8: N_max=  95   N in [20, 30, 35, 45, 55, 60, 70, 80, 85, 95]
  a=0.9: N_max=  85   N in [20, 25, 35, 40, 50, 55, 65, 70, 80, 85]
  a=1.0: N_max=  75   N in [20, 25, 30, 40, 45, 50, 55, 65, 70, 75]
  a=1.1: N_max=  70   N in [20, 25, 30, 35, 40, 50, 55, 60, 65, 70]


In [13]:
n_trials_ext = 30
max_attempts_ext = 100

exp_manager_ext = setup_experiment()

print(f"\n{len(pairs_ext)} pairs x {n_trials_ext} trials each = up to "
      f"{len(pairs_ext) * n_trials_ext} percolating networks")

EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 857
  Timestamp: 20260831_171114
  Results directory: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_857


100 pairs x 30 trials each = up to 3000 percolating networks


---
## 13. Run the extended sweep

Same logic as section 4, but over the adaptive grid from section 12. Checkpoints after every
(N, a) pair, so an interrupt doesn't lose the whole sweep.

In [14]:
results_grid_ext = {}
perc_rate_grid_ext = {}

t_start = time.time()
for idx, (N, a) in enumerate(pairs_ext, start=1):
    seed_base = (hash((N, round(a, 3))) % 100000)
    np.random.seed(seed_base)

    R_effs = []
    total_attempts = 0
    while len(R_effs) < n_trials_ext:
        res, attempts = generate_percolating_network(N, a, k_uniform=R0, max_attempts=max_attempts_ext, boundary=BOUNDARY)
        total_attempts += attempts
        if res is None:
            print(f"  [WARNING] N={N} a={a}: gave up after {total_attempts} draws, "
                  f"only found {len(R_effs)}/{n_trials_ext} percolating networks")
            break
        rl = distance_dependent_impedance(res, R0)
        R_effs.append(rl['impedance'])

    rate = len(R_effs) / total_attempts if total_attempts else 0.0
    results_grid_ext[(N, a)] = R_effs
    perc_rate_grid_ext[(N, a)] = rate
    elapsed = time.time() - t_start
    mean_str = f"{np.mean(R_effs):8.1f}" if R_effs else "     n/a"
    print(f"[{idx:3d}/{len(pairs_ext)}] N={N:3d}  a={a:.1f}   {len(R_effs):2d}/{n_trials_ext} percolating   "
          f"({total_attempts:4d} draws, {rate:5.1%} success)   R_eff mean={mean_str} ohm   "
          f"[{elapsed:7.1f}s elapsed]")

    # checkpoint after every pair so a crash/interrupt doesn't lose the whole sweep
    exp_manager_ext.save_json(
        {"pairs_done": idx, "pairs_total": len(pairs_ext),
         "sweep": [{"N": n, "a": a2, "R_eff_values": v, "perc_rate": perc_rate_grid_ext[(n, a2)]}
                   for (n, a2), v in results_grid_ext.items()]},
        "effective_resistance_sweep_ext_CHECKPOINT.json",
    )

print(f"\nExtended sweep complete in {time.time() - t_start:.1f}s.")

  [WARNING] N=20 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  1/100] N= 20  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    0.1s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


  [WARNING] N=35 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  2/100] N= 35  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    0.2s elapsed]


  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


  [WARNING] N=50 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  3/100] N= 50  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    0.5s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


  [WARNING] N=65 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  4/100] N= 65  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    1.0s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


...

[ 96/100] N= 50  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    84.6 ohm   [ 2307.4s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[ 97/100] N= 55  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    62.4 ohm   [ 2330.0s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[ 98/100] N= 60  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    59.6 ohm   [ 2364.8s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[ 99/100] N= 65  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    51.3 ohm   [ 2416.3s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[100/100] N= 70  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    39.8 ohm   [ 2489.5s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json

Extended sweep complete in 2489.5s.


---
## 14. Box plots: R_eff vs N, grouped by a (ragged grid)

Same idea as section 6, but each `a` now has its own N range (the adaptive grid isn't a full
rectangle), so each panel gets its own x-axis.

In [ ]:
# Figure sizing note: this ends up in a document at a fixed text width, so a
# huge figsize is counterproductive -- LaTeX scales 22 inches down to ~6 and the
# fonts shrink with it. A modest figsize plus explicit font sizes keeps the axis
# labels readable at print size. The per-panel x and y labels are replaced by one
# shared pair (supxlabel/supylabel), and sharey lets only the left column carry
# tick labels, which buys the panels themselves more room.
fig, axes = plt.subplots(2, 5, figsize=(13, 7), sharey=True, constrained_layout=True)
axes = axes.flatten()

for ax, a in zip(axes, a_values_ext):
    Ns = grid_ext[a]
    # some (N, a) below the percolation threshold may have 0 trials -- skip those, not the box
    Ns_nonempty = [N for N in Ns if results_grid_ext[(N, a)]]
    if not Ns_nonempty:
        ax.text(0.5, 0.5, "no percolating\ntrials found", ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='gray')
        ax.text(0.04, 0.95, f'a = {a:.1f}', transform=ax.transAxes, ha='left', va='top', fontsize=13,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=2))
        continue
    data = [results_grid_ext[(N, a)] for N in Ns_nonempty]
    ax.boxplot(data, positions=range(len(Ns_nonempty)), widths=0.6)
    # thin the tick labels when a panel is crowded: every other N still shows the
    # trend, and unrotated labels survive being scaled down
    step = 1 if len(Ns_nonempty) <= 6 else 2
    ax.set_xticks(range(0, len(Ns_nonempty), step))
    ax.set_xticklabels(Ns_nonempty[::step], fontsize=10)
    ax.set_yscale('log')
    # panel label inside the axes instead of a title: readable at print size, and the
    # LaTeX caption carries the figure title
    ax.text(0.04, 0.95, f'a = {a:.1f}', transform=ax.transAxes, ha='left', va='top', fontsize=13,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=2))
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(True, alpha=0.3, which='both')

fig.supxlabel('N (number of sticks)', fontsize=14)
fig.supylabel('R_eff (Ω, log scale)', fontsize=14)
plt.show()
exp_manager_ext.save_figure(fig, "box_R_eff_vs_N_by_a_ext.png")

---
## 15. R_eff(N, a) and percolation rate over the adaptive grid

The grid isn't a rectangle anymore, so a matrix heatmap doesn't apply -- these are scatter plots
over the actual (N, a) points sampled, colored by mean R_eff and by percolation success rate.

In [ ]:
Ns_all = np.array([N for (N, a) in pairs_ext])
as_all = np.array([a for (N, a) in pairs_ext])
means_all = np.array([np.mean(results_grid_ext[(N, a)]) if results_grid_ext[(N, a)] else np.nan
                       for (N, a) in pairs_ext])
rates_all = np.array([perc_rate_grid_ext[(N, a)] for (N, a) in pairs_ext])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc1 = axes[0].scatter(as_all, Ns_all, c=np.log10(means_all), cmap='viridis_r',
                       s=80, edgecolor='k', linewidth=0.3)
axes[0].set_xlabel('a'); axes[0].set_ylabel('N')  # same orientation as the heatmaps
axes[0].set_title('Mean R_eff (log10 ohm)')
fig.colorbar(sc1, ax=axes[0], label='log10(R_eff / ohm)')

sc2 = axes[1].scatter(as_all, Ns_all, c=rates_all, cmap='viridis', vmin=0, vmax=1,
                       s=80, edgecolor='k', linewidth=0.3)
axes[1].set_xlabel('a'); axes[1].set_ylabel('N')
axes[1].set_title('Percolation success rate')
fig.colorbar(sc2, ax=axes[1], label='fraction of draws that percolated')

plt.tight_layout()
# No fig.suptitle here: these figures go into the dissertation, where the
# LaTeX \caption carries the title. An embedded title would duplicate it and
# waste vertical space once the figure is scaled to the text width. The
# per-panel titles below are kept, since they label data (the value of a or
# N for that panel), not the figure.
plt.show()
exp_manager_ext.save_figure(fig, "scatter_R_eff_and_perc_rate_ext.png")

In [ ]:
# Dissertation Figure 8: the same two panels saved separately and titleless
# (the LaTeX subcaptions carry the titles), with a on x and N on y so the figure
# lines up with the Sec. 5.3 heatmaps rather than transposing them.
#
# Two further harmonizations between the panels: points where no draw percolated
# have no R_eff, so they used to vanish from the left panel while still showing
# in the right one; they are now drawn hollow, and every point that fell short of
# n_trials_ext gets its actual count printed, which is the only place in either
# figure where the number of networks per point is not constant.
n_by_pair = {(N, a): len(results_grid_ext[(N, a)]) for (N, a) in pairs_ext}
counts = np.array([n_by_pair[(N, a)] for (N, a) in pairs_ext])
ok = counts > 0

def _annotate_counts(ax):
    for x, y, n in zip(as_all, Ns_all, counts):
        if n < n_trials_ext:
            ax.annotate(f"{n}", (x, y), textcoords="offset points", xytext=(0, 7),
                        ha="center", fontsize=6.5)
    ax.set_xlabel('a'); ax.set_ylabel('N')
    ax.grid(True, alpha=0.25)

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(as_all[ok], Ns_all[ok], c=np.log10(means_all[ok]), cmap='viridis_r',
                s=80, edgecolor='k', linewidth=0.3)
ax.scatter(as_all[~ok], Ns_all[~ok], facecolors='none', edgecolors='k', linewidth=0.8,
           s=80, label='no percolating draw')
_annotate_counts(ax); ax.legend(loc='lower right', fontsize=8)
fig.colorbar(sc, ax=ax, label='log10(R_eff / ohm)')
plt.tight_layout()
exp_manager_ext.save_figure(fig, 'mean_reff_Na_plane.png')
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(as_all, Ns_all, c=rates_all, cmap='viridis', vmin=0, vmax=1,
                s=80, edgecolor='k', linewidth=0.3)
_annotate_counts(ax)
fig.colorbar(sc, ax=ax, label='fraction of draws that percolated')
plt.tight_layout()
exp_manager_ext.save_figure(fig, 'percolation_rate_Na_plane.png')
plt.close(fig)


---
## 16. R_eff collapse vs N*a^2

With ~20 points spread over N and a independently, we can check whether R_eff is really a
function of the combined variable $N \cdot a^2$ (the stick-percolation excluded-area parameter),
as the notes in section 10 above assumed from just 12 points.

In [17]:
na2 = Ns_all * as_all**2
valid = ~np.isnan(means_all)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(na2[valid], means_all[valid], c=as_all[valid], cmap='plasma',
                 s=45, edgecolor='k', linewidth=0.3)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('N * a^2'); ax.set_ylabel('mean R_eff (ohm)')
ax.grid(True, alpha=0.3, which='both')
fig.colorbar(sc, ax=ax, label='a')
plt.tight_layout()
plt.show()
exp_manager_ext.save_figure(fig, "scatter_R_eff_vs_Na2_ext.png")

  [OK] Figure saved: scatter_R_eff_vs_Na2_ext.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_3571/957608969.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_857/scatter_R_eff_vs_Na2_ext.png'

---
## 17. Save consolidated summary

In [18]:
sweep_records_ext = []
for (N, a) in pairs_ext:
    vals = results_grid_ext[(N, a)]
    sweep_records_ext.append({
        "N": N, "a": a, "n": len(vals),
        "mean_ohm": float(np.mean(vals)) if vals else None,
        "median_ohm": float(np.median(vals)) if vals else None,
        "std_ohm": float(np.std(vals)) if vals else None,
        "perc_rate": perc_rate_grid_ext[(N, a)],
        "R_eff_values": vals,
    })

summary_ext = {
    "experiment": exp_manager_ext.get_info(),
    "config": {
        "pairs": pairs_ext, "n_trials": n_trials_ext, "R0": R0,
        "max_attempts": max_attempts_ext, "boundary": BOUNDARY,
        "grid_design": "adaptive: N_max(a) chosen so estimated node count (~0.5*N^2*a^2) stays <= node_cap=3000",
    },
    "sweep": sweep_records_ext,
}

exp_manager_ext.save_json(summary_ext, "effective_resistance_sweep_summary_ext.json")

print(f"\nAll results saved to: {exp_manager_ext.experiment_dir}")
for f in sorted(os.listdir(exp_manager_ext.experiment_dir)):
    print(f"  - {f}")

  [OK] JSON saved: effective_resistance_sweep_summary_ext.json

All results saved to: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_857
  - box_R_eff_vs_N_by_a_ext.png
  - effective_resistance_sweep_ext_CHECKPOINT.json
  - effective_resistance_sweep_summary_ext.json
  - scatter_R_eff_and_perc_rate_ext.png
  - scatter_R_eff_vs_Na2_ext.png
